# Task 3 – Statistical tests
**Who:** Joud & Maram  
**Input:** `food_final.csv`, the final dataset for the model. 

Every test follows **Hypotheses → Assumptions → Test → Result → Interpretation**, with α = 0.05.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

pandas and numpy for the tables, scipy.stats for the tests (as in Lab 06), matplotlib for the plots.

In [2]:
final = pd.read_csv('food_final.csv')
print(final.shape)

(126787, 53)


We read the final dataset file. (Write the number of rows and columns here)

## Statistical tests
For the tests that compare **before and after** the change we use only **Pre** (before 1 July 2018) and **Post** (after 30 June 2019), and leave out the transition year, which is a mix of both. The transition year is tested separately in H4.

In [3]:
pre_post = final[final['Period'].isin(['Pre', 'Post'])]
pre_post['Period'].value_counts()

Period
Pre     64293
Post    47780
Name: count, dtype: int64

Number of Pre and Post inspections used in the before/after tests. (Write here the two numbers.)

**Effect size for chi-square tests (Cramér's V).** With more than 100,000 rows almost every difference is significant, so we also report how **big** it is. Cramér's V = √(χ² / (n × (k − 1))), where k is the smaller number of rows or columns of the table. It goes from 0 (no relationship) to 1 (perfect relationship); about 0.1 = small, 0.3 = medium, 0.5 = large. (This formula is new for us; it is the usual effect size for chi-square.)

In [4]:
def cramers_v(table):
    chi2, p, dof, expected = stats.chi2_contingency(table)
    n = table.values.sum()
    k = min(table.shape) - 1
    return np.sqrt(chi2 / (n * k))

A small function that calculates Cramér's V from a crosstab, so we can use it for every chi-square test.

### H1a – Did the distribution of results change?
**H0:** the shares of Pass / Pass w/ Conditions / Fail are the same before and after the change.  
**H1:** the shares are different.  
**Assumptions:** two categorical variables (Period, Results), independent inspections, every expected count ≥ 5.  
**Test:** chi-square test of independence.

In [5]:
table_h1a = pd.crosstab(pre_post['Period'], pre_post['Results'])
table_h1a

Results,Fail,Pass,Pass w/ Conditions
Period,,,
Post,11343,23308,13129
Pre,13905,42441,7947


The counts of each result before and after.

In [6]:
pd.crosstab(pre_post['Period'], pre_post['Results'], normalize='index').round(3)

Results,Fail,Pass,Pass w/ Conditions
Period,,,
Post,0.237,0.488,0.275
Pre,0.216,0.660,0.124


The same as shares (each row adds up to 1). (Write here how each share changes from Pre to Post.)

In [7]:
chi2, p, dof, expected = stats.chi2_contingency(table_h1a)
print(f'Chi-square = {chi2:.2f}, df = {dof}, p-value = {p:.3e}, min expected = {expected.min():.0f}')
print(f"Cramer's V = {cramers_v(table_h1a):.3f}")

Chi-square = 4772.35, df = 2, p-value = 0.000e+00, min expected = 8985
Cramer's V = 0.206


**Result and interpretation:** (Write here: reject or fail to reject H0, Cramér's V and its size, and what it means in words.)

### H1b – Did the number of violations change?
**H0:** the number of violations per inspection has the same distribution before and after.  
**H1:** the distributions are different.  
**Assumptions:** two independent groups, a numeric variable. The counts are not normal (right-skewed, many zeros, see the histogram in Task 1), so we use the non-parametric **Mann–Whitney U** test, as in Lab 06 and Assignment 2.

In [8]:
pre_post.groupby('Period')[['Num_Violations', 'Num_Critical', 'Num_Serious']].median()

,Num_Violations,Num_Critical,Num_Serious
Period,,,
Post,3.0,0.0,0.0
Pre,3.0,0.0,0.0


Median number of violations before and after. (Write here.)

In [9]:
pre_post.groupby('Period')[['Num_Violations', 'Num_Critical', 'Num_Serious']].mean().round(2)

,Num_Violations,Num_Critical,Num_Serious
Period,,,
Post,4.25,0.20,0.91
Pre,3.45,0.24,0.36


Mean number of violations before and after. (Write here.)

In [10]:
pre_v = pre_post[pre_post['Period'] == 'Pre']['Num_Violations']
post_v = pre_post[pre_post['Period'] == 'Post']['Num_Violations']
u_stat, u_p = stats.mannwhitneyu(pre_v, post_v, alternative='two-sided')
rank_biserial = 2 * (u_stat / (len(pre_v) * len(post_v))) - 1
print(f'Mann-Whitney U = {u_stat:.0f}, p-value = {u_p:.3e}, rank-biserial = {rank_biserial:.3f}')

Mann-Whitney U = 1438185468, p-value = 1.962e-75, rank-biserial = -0.064


**Result and interpretation:** (Write here.) The rank-biserial correlation is the effect size (same formula as in Assignment 2): a negative value means Pre inspections tend to have **fewer** violations than Post inspections.

In [11]:
for col in ['Num_Critical', 'Num_Serious']:
    u, p_col = stats.mannwhitneyu(pre_post[pre_post['Period'] == 'Pre'][col], pre_post[pre_post['Period'] == 'Post'][col], alternative='two-sided')
    print(f'{col}: U = {u:.0f}, p-value = {p_col:.3e}')

Num_Critical: U = 1573838368, p-value = 2.956e-29
Num_Serious: U = 1234688238, p-value = 0.000e+00


The same test for critical and serious violations separately. (Write here. From Task 2 we expect critical to change little and serious to change a lot, because the new Priority Foundation level is much wider than the old Serious level.)

### H1c – Did the mix of violation topics change?
**H0:** the share of inspections with each topic (rodents, hand washing, …) is the same before and after.  
**H1:** the share is different.  
**Test:** one chi-square test per topic (Period × topic yes/no).

In [12]:
topics = ['has_rodents', 'has_temperature', 'has_handwashing', 'has_toxic', 'has_water_sewage', 'has_dirty_surfaces',
          'has_thermometers', 'has_food_handler', 'has_smoking', 'has_summary_report']
rows = []
for t in topics:
    table = pd.crosstab(pre_post['Period'], pre_post[t])
    chi2_t, p_t, dof_t, exp_t = stats.chi2_contingency(table)
    rows.append({'Topic': t,
                 'Pre share': pre_post[pre_post['Period'] == 'Pre'][t].mean().round(3),
                 'Post share': pre_post[pre_post['Period'] == 'Post'][t].mean().round(3),
                 'Chi-square': round(chi2_t, 1), 'p-value': p_t, "Cramer's V": round(cramers_v(table), 3)})
h1c = pd.DataFrame(rows).sort_values("Cramer's V", ascending=False)
h1c

,Topic,Pre share,Post share,Chi-square,p-value,Cramer's V
2,has_handwashing,0.037,0.206,8037.7,0.000000e+00,0.268
5,has_dirty_surfaces,0.676,0.532,2396.2,0.000000e+00,0.146
0,has_rodents,0.109,0.161,653.2,4.462346e-144,0.076
4,has_water_sewage,0.013,0.033,510.9,4.058282e-113,0.068
9,has_summary_report,0.011,0.020,152.3,5.409584e-35,0.037
3,has_toxic,0.001,0.004,131.1,2.389174e-30,0.034
7,has_food_handler,0.116,0.132,66.9,2.787100e-16,0.024
1,has_temperature,0.081,0.071,41.9,9.661531e-11,0.019
8,has_smoking,0.002,0.002,7.9,4.902536e-03,0.008
6,has_thermometers,0.086,0.084,1.2,2.705085e-01,0.003


One row per topic, sorted by effect size (biggest change first).  
**Note on many tests:** we run 10 tests, so with α = 0.05 one of them could be significant just by chance (a Type I error). We therefore focus on the topics with a clear effect size, not only on the p-values.  
**Result and interpretation:** (Write here which topics changed the most.)

### H1 – Conclusion
(Write here one short paragraph that combines H1a, H1b and H1c: for example, the fail rate changed little, but the way results and violations are recorded changed a lot.)

### H4 – Was the 90-day grace period different from the rest of the transition year?
**H0:** the shares of Pass / Pass w/ Conditions / Fail are the same in the grace period (1 July – 30 September 2018) and in the rest of the transition year.  
**H1:** they are different.  
**Test:** chi-square test of independence.

In [13]:
transition = final[final['Period'].isin(['Grace (90 days)', 'Late transition'])]
table_h4 = pd.crosstab(transition['Period'], transition['Results'])
pd.crosstab(transition['Period'], transition['Results'], normalize='index').round(3)

Results,Fail,Pass,Pass w/ Conditions
Period,,,
Grace (90 days),0.217,0.124,0.659
Late transition,0.209,0.239,0.551


Shares of each result in the two parts of the transition year. (Write here. In Task 1 we saw about 12% clean Pass in the grace period vs about 24% after.)

In [14]:
chi2, p, dof, expected = stats.chi2_contingency(table_h4)
print(f'Chi-square = {chi2:.2f}, df = {dof}, p-value = {p:.3e}')
print(f"Cramer's V = {cramers_v(table_h4):.3f}")

Chi-square = 213.60, df = 2, p-value = 4.141e-47
Cramer's V = 0.120


**Result and interpretation:** (Write here.)

### H5 – Does a business's previous result help predict its current result?
**H0:** failing the previous inspection and failing the current inspection are not related.  
**H1:** they are related.  
**Assumptions:** we use only **normal visits** (no re-inspections), because a re-inspection right after a fail is usually a pass (Maram's notebook), and only businesses that have a previous visit.  
**Test:** chi-square test of independence.

In [15]:
normal = final[(final['Is_Reinspection'] == 0) & (final['First_Inspection'] == 0)]
table_h5 = pd.crosstab(normal['Prev_Fail'], normal['Fail'])
pd.crosstab(normal['Prev_Fail'], normal['Fail'], normalize='index').round(3)

Fail,0,1
Prev_Fail,,
0.0,0.749,0.251
1.0,0.666,0.334


Fail rate now, for businesses that passed (0) or failed (1) at their previous visit. (Write here.)

In [16]:
chi2, p, dof, expected = stats.chi2_contingency(table_h5)
print(f'Chi-square = {chi2:.2f}, df = {dof}, p-value = {p:.3e}')
print(f"Cramer's V = {cramers_v(table_h5):.3f}")

Chi-square = 23.25, df = 1, p-value = 1.425e-06
Cramer's V = 0.017


**Result and interpretation:** (Write here. If p < 0.05, the past helps predict the future, which supports using the history columns in the model, even though the city says the data is "not intended to predict future findings".)

### Risk level
The chi-square test for Risk × Fail is in `Extra.ipynb` (χ² = 72.80, p ≈ 1.6 × 10⁻¹⁶, Spearman ρ = 0.013): the fail rate differs between risk levels, but the relationship is very weak, and part of it is explained by the much higher share of License inspections in Risk 3 (37.6% vs 11.2% in Risk 1).

## Summary of the results
(Write here a table: hypothesis, test, statistic, p-value, effect size, decision. This table goes into the brief report.)